# Load FOCUS cost export (CSV variant of FCA's `01_Load_Focus`)

FCA's native `01_Load_Focus.Notebook` reads the **Parquet** output of a Cost Management
FOCUS export. If your export is configured for **CSV** instead (same FOCUS schema, different
file format — check your export's Datasets tab in Cost Management), that notebook's
`spark.read.parquet(...)` call will never find anything.

This notebook is FCA's `01_Load_Focus.Notebook` logic, unchanged except for the file format:
same wildcard/month-folder path detection, same `focus_staging` → `focus` Delta table target.
Once `focus` is populated, **FCA's existing `01_Load_Focus_Fabric`, semantic model, and
report all run on top of it unchanged** — no need to touch anything downstream.

Using the existing export instead of a live API pull also sidesteps Cost Management API
throttling entirely, since this only reads files that already landed in storage — see
[`../notebook-api-ingestion/README.md`](../notebook-api-ingestion/README.md) for context on
why that mattered here.

**Prerequisite**: a OneLake **shortcut** on this Lakehouse pointing at the export's storage
container — same mechanism as [`../../Deploy.md`](../../Deploy.md) section 1.1, just pointed
at your CSV export's container/directory instead of a FOCUS Parquet one.

## Step 0 – Verify your `ServiceName` value before filtering

In [ ]:
from delta.tables import *
from notebookutils import mssparkutils
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import col, when, from_json, date_format, lit, row_number, max, lower
from pyspark.sql.types import StructType, StringType
from pyspark.sql.window import Window
import re
import glob

fromMonth = 0
toMonth = 0
rawSourcePath = "Files/focuscost"  # your shortcut's path under Files/

# FOCUS's ServiceName for Fabric has been observed as both "Microsoft Fabric" (FOCUS spec
# convention, no dot) and "Microsoft.Fabric" (Azure resource-provider convention, with a
# dot) depending on export version/tenant -- confirm which one you actually have below
# before relying on the hardcoded filter further down.
fabric_service_names = ["Microsoft Fabric", "Microsoft.Fabric"]

In [ ]:
# One-time check: point this at a single known CSV file from your export (any date folder)
# to confirm the actual ServiceName value(s) present before the real ingestion loop filters on it.
sample_csv_path = "<Files/focuscost/.../a-single-export-file>.csv"  # fill in, then run once

sample_df = spark.read.option("header", "true").csv(sample_csv_path)
sample_df.select("ServiceName").distinct().show(truncate=False)

Adjust `fabric_service_names` above if the distinct value(s) shown don't match, then continue.

## FUNCTIONS

Identical to FCA's `01_Load_Focus.Notebook`, except `find_first_csv_file` looks for `.csv`
instead of `.parquet`.

In [ ]:
def find_first_csv_file(path):
    try:
        for entry in mssparkutils.fs.ls(path):
            if entry.isFile and entry.name.endswith(".csv"):
                return entry.path
            elif entry.isDir:
                result = find_first_csv_file(entry.path)
                if result:
                    return result
    except Exception as e:
        print(f"Error accessing {path}: {e}")
    return None

def generate_wildcard_path(full_path: str, raw_source_path: str, current_Date_Folder: str, snapshot_folder: str) -> str:
    idx = full_path.find(raw_source_path)
    if idx == -1:
        raise ValueError("rawSourcePath not found in full path")

    base_uri = full_path[:idx]
    detailPath = full_path[idx+len(rawSourcePath):]

    idx = detailPath.find(current_Date_Folder)
    detailPreMonth = detailPath[:idx]
    detailPostMonth = detailPath[idx+len(current_Date_Folder)-1:]

    startleveltoAddPre = detailPreMonth.count('/')
    startleveltoAddPost = detailPostMonth.count('/')

    wildcard_path = f"{base_uri}{raw_source_path}{'/*' * startleveltoAddPre}/{snapshot_folder}{'/*' * startleveltoAddPost}.csv"
    return wildcard_path

def generateArrayOFPeriod(from_Month: int, to_month: int):
    today = datetime.today()
    yesterday = today - timedelta(days=1)
    first_day = yesterday.replace(day=1)

    periodToLoad = []
    if from_Month == to_month:
        periodDate = first_day + relativedelta(months=to_month)
        periodToLoad.append(periodDate.date())
    else:
        for i in range(from_Month, to_month+1):
            periodDate = first_day + relativedelta(months=i)
            periodToLoad.append(periodDate.date())

    return periodToLoad

## STEP 1 – Load Silver

Same structure-detection and month-folder logic as FCA's original notebook, reading CSV
instead of Parquet. `header=True` reads FOCUS's column names directly from the file; FOCUS
CSV exports can have embedded commas/quotes in fields like `Tags`, so `multiLine`/`escape`
are set defensively.

In [ ]:
structurePath = find_first_csv_file(rawSourcePath)
periodsToLoad = generateArrayOFPeriod(fromMonth, toMonth)

current_Date_Folder = ""
print(f"Analyze structurePath to find date pattern: {structurePath}")
match = re.search(r"\/[1-2][0-9][0-9][0-9]\/[0-1][0-9]\/", structurePath)
if match:
    current_Date_Folder = match.group()
    print(f"Find monthly date: {current_Date_Folder}")
    date_pattern = "YYYY/MM"
else:
    match = re.search(r"\/[1-2][0-9][0-9][0-9][0-1][0-9][0-3][0-9]-[1-2][0-9][0-9][0-9][0-1][0-9][0-3][0-9]\/", structurePath)
    if match:
        current_Date_Folder = match.group()
        print(f"Find monthly date: {current_Date_Folder}")
        date_pattern = "YYYYMMDD-YYYYMMDD"

if (current_Date_Folder == ""):
    raise ValueError("No Month Pattern found in structurePath -- your export's folder layout differs from FCA's assumed YYYY/MM or YYYYMMDD-YYYYMMDD pattern; adjust the regexes above to match it.")

In [ ]:
for per in periodsToLoad:
    print("Start Period : " + per.strftime("%Y-%m-%d"))

    spark.sql("DROP TABLE IF EXISTS focus_staging")

    if date_pattern == "YYYY/MM":
        snapshot_folder = per.strftime("%Y/%m")
    else:
        fromFormatedDate = per.strftime("%Y%m%d")
        toFormatedDate = (per + relativedelta(months=1) + relativedelta(days=-1)).strftime("%Y%m%d")
        snapshot_folder = fromFormatedDate + "-" + toFormatedDate

    wildcard = generate_wildcard_path(structurePath, rawSourcePath, current_Date_Folder, snapshot_folder)
    print("Used path to load data: " + wildcard)

    idx = wildcard.find(rawSourcePath)
    base_uri = wildcard[:idx]
    relative_uri = wildcard[idx:]
    glob_wildcard = f"/lakehouse/default/{relative_uri}"
    print("Used path to glob discover: " + glob_wildcard)

    lenBase = len("/lakehouse/default/")
    folder_paths = glob.glob(glob_wildcard)

    find_files = len(folder_paths)
    full_files_paths = [f"{base_uri}{path[lenBase:]}" for path in folder_paths]
    print(f"Found {find_files} file(s) path(s) to load")

    if find_files > 0:
        try:
            df = (
                spark.read
                .option("header", "true")
                .option("multiLine", "true")
                .option("escape", '"')
                .csv(full_files_paths)
            )
            df.write.format('delta').mode('overwrite').option("overwriteSchema", "true").saveAsTable("focus_staging")

            df = spark.sql("SELECT BillingPeriodStart FROM focus_staging LIMIT 1")
            value = df.first()['BillingPeriodStart']

            if spark.catalog.tableExists("focus"):
                print("Table exists, snapshot will be cleaned.")
                spark.sql(f"DELETE FROM focus WHERE BillingPeriodStart = '{value}'")

            focus_staging_df = DeltaTable.forPath(spark, "Tables/focus_staging").toDF()
            focus_staging_df.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable("focus")

        except AnalysisException as e:
            if "PATH_NOT_FOUND" in str(e):
                print(f"Path not found: {wildcard}")
            else:
                raise
    else:
        print(f"No files path found from wildcard path: {wildcard}")

    print("End Period : " + per.strftime("%Y-%m-%d"))

## Next steps

- Run FCA's existing `01_Load_Focus_Fabric.Notebook` unchanged — it reads from the `focus`
  table this notebook just populated, and filters to Fabric costs. **Before your first real
  run**, update its `ServiceName = 'Microsoft.Fabric'` filter if Step 0 above showed your
  export actually uses `'Microsoft Fabric'` (no dot) instead.
- From there, FCA's semantic model (`FCA_Core_SM`) and report (`FCA_Core_Report`) work as
  documented in [`../../Deploy.md`](../../Deploy.md) — no changes needed.
- Schedule this notebook the same way FCA schedules `01_Load_Focus` (via the `Load FCA E2E`
  pipeline, or its own notebook Schedule) — since it only reads already-exported files, it's
  safe to run as often as your export refreshes, with none of the Cost Management API
  throttling the live-query approach in `../notebook-api-ingestion/` ran into.